# Credit Default - Track A

Thin orchestration notebook for S02 only. Reusable logic stays under `src/credit_default/`.


## S02 Gate 2 - Data Loading, Quality, EDA, Split, Preprocessing

This notebook loads the UCI dataset, reports schema and EDA outputs, shows reproducible tables/plots, and prepares deterministic train/validation/test splits.


In [ ]:
from IPython.display import display

import pandas as pd

from credit_default.data.eda import (
    build_eda_report,
    plot_default_rate_by_category,
    plot_numeric_distributions,
    plot_target_distribution,
)
from credit_default.data.load import load_uci_dataset
from credit_default.data.schema import build_schema_report
from credit_default.data.split import make_splits, summarize_splits
from credit_default.features.preprocessing import describe_preprocessor, fit_preprocessor


In [ ]:
features, target = load_uci_dataset()
schema_report = build_schema_report(features, target)
eda_report = build_eda_report(features, target)
splits = make_splits(features, target)
preprocessor_bundle = fit_preprocessor(splits.X_train)
split_summary = pd.DataFrame.from_dict(summarize_splits(splits), orient='index').reset_index(names='split')
schema_summary = pd.DataFrame([
    {
        'rows': schema_report.n_rows,
        'features': schema_report.n_features,
        'target_name': schema_report.target_name,
        'duplicate_rows': schema_report.duplicate_rows,
        'schema_issues': ', '.join(schema_report.schema_issues) or 'none',
        'leakage_risks': ', '.join(schema_report.leakage_risks) or 'none',
        'target_positive_rate': schema_report.target_positive_rate,
    }
])
preprocessing_summary = pd.DataFrame([describe_preprocessor(preprocessor_bundle)])


In [ ]:
display(schema_summary)
display(eda_report.target_distribution)
display(split_summary)
display(preprocessing_summary)


In [ ]:
display(eda_report.descriptive_stats)
display(eda_report.numeric_distribution_summary)
display(eda_report.missing_values if not eda_report.missing_values.empty else pd.DataFrame([{'column': 'none', 'missing_count': 0, 'missing_rate': 0.0}]))
display(eda_report.duplicate_summary)
display(eda_report.inconsistent_values if not eda_report.inconsistent_values.empty else pd.DataFrame([{'column': 'none', 'issue': 'none', 'details': 'none'}]))


In [ ]:
for column, table in eda_report.default_rate_by_category.items():
    print(f'Default rate by {column}')
    display(table)

for column, table in eda_report.default_rate_by_numeric_bin.items():
    print(f'Default rate by binned {column}')
    display(table)


In [ ]:
plot_target_distribution(eda_report.target_distribution)
plot_numeric_distributions(features, eda_report.selected_numeric_columns)
plot_default_rate_by_category(eda_report.default_rate_by_category)


## S03 - Modeling

Thin orchestration only. Training, tuning, thresholding, comparison, and final evaluation stay in `src/credit_default/modeling/`.


In [ ]:
from credit_default.modeling.train import run_modeling_workflow


In [ ]:
modeling_result = run_modeling_workflow(splits)
display(modeling_result.validation_comparison)
display(modeling_result.champion_selection_comparison)
display(pd.DataFrame([{
    'champion_model': modeling_result.champion_test_result.model_name,
    'variant': 'tuned' if modeling_result.champion_test_result.tuned else 'base',
    'threshold': modeling_result.champion_test_result.threshold,
    'validation_roc_auc': modeling_result.champion_validation_result.metrics['roc_auc'],
    'validation_f1': modeling_result.champion_validation_result.metrics['f1'],
    'validation_recall': modeling_result.champion_validation_result.metrics['recall'],
    'test_roc_auc': modeling_result.champion_test_result.metrics['roc_auc'],
    'test_f1': modeling_result.champion_test_result.metrics['f1'],
    'test_recall': modeling_result.champion_test_result.metrics['recall'],
}]))
display(pd.DataFrame(modeling_result.champion_test_result.confusion_matrix, columns=['pred_0', 'pred_1'], index=['true_0', 'true_1']))
